# Stage 2.2 — Māori Data Evidence Extraction

Takes records labelled `maori_origin` in Stage 2.1 (Hugging Face `dinushiTJ/nz_research_commons_classification`), retrieves the full text from **University of Waikato Research Commons**, and determines whether each paper actually contains Māori data, what kind, and where the evidence is.

**Pipeline**

1. Load Stage 2.1 labels from Hugging Face → pick Māori-labelled records
2. Find each record in Research Commons → download the open-access PDF
3. Extract text, fix macrons (`Maori ¯`, `Mäori`, `Maaori`), split into section-aware chunks
4. Keyword retrieval → select chunks (opening pages + Methods/Results/Ethics always kept)
5. Gemini structured extraction (flags + verbatim evidence quotes)
6. Validate every quote against the source text (anti-hallucination)
7. Deterministic rules → final category + CONFIRM/REVIEW route + governance flag
8. Save the Māori-data inventory

**Before running:** add `GEMINI_API_KEY` in Colab → 🔑 Secrets (left sidebar) and enable notebook access. Without a key, the notebook falls back to the pilot extractions for the two default papers so you can still run every other step.

> Outputs are decision support for review, not authoritative classifications. Only study-level information is extracted — no participant-level, whakapapa, wāhi tapu location or restricted mātauranga content.

## 0. Setup

In [1]:
!pip -q install pymupdf rapidfuzz pydantic google-genai pyarrow pandas requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.4 MB/s eta 0:00:00


In [2]:
import json, re, time, copy, unicodedata, hashlib
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Literal, Optional

import pandas as pd
import requests
import pymupdf
from rapidfuzz import fuzz
from pydantic import BaseModel, Field

OUT = Path("/content/stage2_2") if Path("/content").exists() else Path("stage2_2_out")
(OUT / "pdfs").mkdir(parents=True, exist_ok=True)
(OUT / "chunks").mkdir(parents=True, exist_ok=True)
print("Output folder:", OUT)

Output folder: /content/stage2


## 1. Configuration

In [3]:
N_PAPERS = 2                       # how many Māori-labelled records to process
HF_DATASET = "dinushiTJ/nz_research_commons_classification"
HF_SPLIT = "test"                  # "train" or "test"

# Leave empty to auto-pick; or list titles to process specific papers
# (these two are the pilot papers: one with participant data, one with cultural/policy material)
PICK_TITLES = [
    "Wāhine Māori entrepreneurs and business growth",
    "Māori provisions in plans: Mana whenua, mauri of water, and wahi tapu",
]

GEMINI_MODEL = "gemini-2.5-flash"
CHUNK_CHARS = 3500                 # ~900 tokens per chunk
CONTEXT_BUDGET_CHARS = 60000       # max text sent to Gemini per paper (~15k tokens)
QUOTE_MATCH_THRESHOLD = 90         # fuzzy match score needed for a quote to count as verified

RC_API = "https://researchcommons.waikato.ac.nz/server/api"
HEADERS = {"User-Agent": "maori-data-stage2_2-research/0.1"}

In [4]:
# Gemini key from Colab Secrets (or environment)
import os
GEMINI_API_KEY = None
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
USE_GEMINI = bool(GEMINI_API_KEY)
print("Gemini enabled:", USE_GEMINI)

Gemini enabled: True


## 2. Load Stage 2.1 labels from Hugging Face

In [5]:
url = f"https://huggingface.co/api/datasets/{HF_DATASET}/parquet/default/{HF_SPLIT}/0.parquet"
cols = ["title", "authors", "year", "subjects", "abstract", "record_id_hash",
        "classification_label", "classification_confidence", "classification_reason"]
df = pd.read_parquet(url, columns=cols)
# ClassLabel: 0 = maori_origin, 1 = non_maori_origin
df["stage2_1_label"] = df["classification_label"].map({0: "maori_origin", 1: "non_maori_origin"})
maori = df[df.stage2_1_label == "maori_origin"].copy()
print(df.stage2_1_label.value_counts(), "\n")
maori[["year", "title", "classification_confidence"]].head(15)

stage1_label
non_maori_origin    2951
maori_origin         367
Name: count, dtype: int64 



,year,title,classification_confidence
7,2021,Ahumoana tawhito (ancient aquaculture): The tr...,1.00
10,2023,When loanwords are not lone words: Using netwo...,0.90
17,2014,Te Tuhirau i Rehu i Ringa: Translating Sacred ...,1.00
22,2004,PIWeCS: enhancing human/machine agency in an i...,0.90
23,2009,Revitalizing Te Ika-a-Maui: Māori migration an...,1.00
38,2016,Hoera Te Mimiha,1.00
41,2023,Māori and Pacific language data in linguistics...,0.95
52,2017,Te tuhirau kōrero,1.00
57,2007,Renewal and resistance: Moko in contemporary N...,1.00
62,2020,Participatory design in Māori cultural contexts,1.00


## 3. Find papers in Research Commons and download PDFs

In [6]:
def rc_search(title: str, size: int = 3) -> list[dict]:
    # Solr treats : ( ) [ ] " etc. as query syntax, e.g. "plans: Mana" becomes a field query
    q = re.sub(r'[:()\[\]{}"^~*?\\/!+\-|&]', " ", title)
    r = requests.get(f"{RC_API}/discover/search/objects", headers=HEADERS, timeout=30,
                     params={"query": q, "dsoType": "ITEM", "size": size})
    r.raise_for_status()
    objs = r.json()["_embedded"]["searchResult"]["_embedded"]["objects"]
    out = []
    for o in objs:
        it = o["_embedded"]["indexableObject"]; md_ = it["metadata"]
        out.append({"uuid": it["uuid"], "handle": it.get("handle"),
                    "rc_title": md_.get("dc.title", [{}])[0].get("value", ""),
                    "type": md_.get("dc.type", [{}])[0].get("value", "")})
    return out

def rc_pdf_bitstreams(item_uuid: str) -> list[dict]:
    b = requests.get(f"{RC_API}/core/items/{item_uuid}/bundles", headers=HEADERS, timeout=30).json()
    pdfs = []
    for bundle in b["_embedded"]["bundles"]:
        if bundle["name"] != "ORIGINAL":
            continue
        bs = requests.get(bundle["_links"]["bitstreams"]["href"], headers=HEADERS, timeout=30).json()
        for s in bs["_embedded"]["bitstreams"]:
            if s["name"].lower().endswith(".pdf"):
                pdfs.append({"bitstream": s["uuid"], "name": s["name"], "size": s["sizeBytes"]})
    return pdfs

def title_match(a: str, b: str) -> float:
    return fuzz.token_set_ratio(fold(a), fold(b))

def fold(text: str) -> str:
    """Macron-insensitive lowercase form, used only for matching."""
    t = unicodedata.normalize("NFD", text or "")
    t = "".join(c for c in t if unicodedata.category(c) != "Mn")
    t = re.sub(r"\s+", " ", t).lower().strip()
    return re.sub(r"([aeiou])\1", r"\1", t)     # Maaori -> maori (Waikato double vowels)

In [7]:
# choose candidate records
if PICK_TITLES:
    cand = pd.concat([maori[maori.title.apply(lambda t: title_match(t, q) >= 90)].head(1)
                      for q in PICK_TITLES])
else:
    cand = maori.sort_values("classification_confidence", ascending=False)

papers = []
for _, row in cand.iterrows():
    if len(papers) >= N_PAPERS:
        break
    try:
        hits = [h for h in rc_search(row.title) if title_match(h["rc_title"], row.title) >= 90]
        if not hits:
            print("✗ not found in Research Commons:", row.title[:70]); continue
        pdfs = rc_pdf_bitstreams(hits[0]["uuid"])
        if not pdfs:
            print("✗ no open-access PDF:", row.title[:70]); continue
        pid = f"p{len(papers)+1:03d}"
        path = OUT / "pdfs" / f"{pid}.pdf"
        r = requests.get(f"{RC_API}/core/bitstreams/{pdfs[0]['bitstream']}/content",
                         headers=HEADERS, timeout=120)
        r.raise_for_status(); path.write_bytes(r.content)
        papers.append({"paper_id": pid, "title": row.title, "year": row.year,
                       "record_id_hash": row.record_id_hash, "stage2_1_label": row.stage2_1_label,
                       "stage2_1_confidence": float(row.classification_confidence),
                       "handle": hits[0]["handle"], "rc_type": hits[0]["type"],
                       "pdf_path": str(path), "pdf_sha256": hashlib.sha256(r.content).hexdigest()})
        print(f"✓ {pid} | {hits[0]['type']:<24} | {row.title[:70]}")
        time.sleep(1)          # be polite to the repository
    except Exception as e:
        print("✗ error:", row.title[:60], "|", e)

pd.DataFrame(papers)[["paper_id", "year", "rc_type", "handle", "title"]]

✓ p001 | Conference Contribution  | Wāhine Māori entrepreneurs and business growth: The power of social an
✓ p002 | Technical Report         | Māori provisions in plans: Mana whenua, mauri of water, and wahi tapu


,paper_id,year,rc_type,handle,title
0,p001,2024,Conference Contribution,10289/17973,Wāhine Māori entrepreneurs and business growth...
1,p002,2008-07-31,Technical Report,10289/6103,"Māori provisions in plans: Mana whenua, mauri ..."


## 4. Text extraction, macron normalisation and section-aware chunking

Old NZ documents often encode macrons differently. The pilot found `Maori ¯` (detached macron from PDF extraction), `Mäori` (diaeresis fonts, pre-2010 reports) and `Maaori` (Waikato-Tainui double vowels). All are normalised so keyword search and quote validation work.

In [8]:
MACRON = dict(zip("aeiouAEIOU", "āēīōūĀĒĪŌŪ"))

def normalise(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"([aeiouAEIOU])\s?\u00af", lambda m: MACRON[m.group(1)], text)   # Maori ¯
    text = text.translate(str.maketrans("äëïöüÄËÏÖÜ", "āēīōūĀĒĪŌŪ"))               # Mäori
    text = text.replace("\u00ad", "")
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    return re.sub(r"[ \t]+", " ", text)

SECTION_RE = re.compile(
    r"^\s*(\d+(\.\d+)*\.?\s+)?(abstract|introduction|background|literature review|"
    r"method(s|ology)?|research design|participants|data( collection)?|analysis|"
    r"results?|findings|discussion|conclusions?|limitations|ethics|acknowledg(e)?ments?|"
    r"funding|data availability|references|bibliography|appendix.*)\s*$", re.I)

@dataclass
class Chunk:
    chunk_id: str
    section: str
    page: int
    text: str

def extract_chunks(pdf_path: str, max_chars: int = CHUNK_CHARS) -> list[Chunk]:
    doc = pymupdf.open(pdf_path)
    chunks, section, buf, buf_page, n = [], "front", [], 1, 0
    def flush():
        nonlocal buf, n
        body = normalise("\n".join(buf)).strip()
        if len(body) > 80:
            n += 1; chunks.append(Chunk(f"c{n:03d}", section, buf_page, body))
        buf = []
    for pno, page in enumerate(doc, start=1):
        for line in page.get_text().splitlines():
            if SECTION_RE.match(line) and len(line) < 60:
                flush(); section = line.strip().lower(); buf_page = pno; continue
            if not buf: buf_page = pno
            buf.append(line)
            if sum(len(l) for l in buf) > max_chars: flush()
    flush()
    out = []
    for c in chunks:                                  # drop reference list onwards
        if re.match(r"(references|bibliography)", c.section): break
        out.append(c)
    return out, len(doc)

for p in papers:
    chunks, n_pages = extract_chunks(p["pdf_path"])
    p["chunks"], p["n_pages"] = chunks, n_pages
    chars = sum(len(c.text) for c in chunks)
    p["needs_ocr"] = chars < 500 * max(n_pages, 1) * 0.1
    print(f"{p['paper_id']}: {n_pages} pages, {len(chunks)} chunks, {chars:,} chars"
          + ("  ⚠ low text — may need OCR" if p["needs_ocr"] else ""))

p001: 22 pages, 11 chunks, 32,496 chars
p002: 71 pages, 58 chunks, 201,286 chars


## 5. Candidate retrieval

Keyword lexicon (macron- and double-vowel-insensitive). Always kept: the first 6 pages (abstract / preface / purpose — the pilot showed keyword ranking alone skipped the preface that documented Māori leadership) plus any Methods, Results, Ethics, Acknowledgements, Funding or Data sections.

In [9]:
LEXICON = ["maori", "iwi", "hapu", "whanau", "marae", "tikanga", "matauranga", "whakapapa",
           "kaupapa maori", "rangatahi", "kaumatua", "wahine", "tangata whenua", "mana whenua",
           "te tiriti", "treaty", "waitangi", "te reo", "ethnicity", "self-identif",
           "participant", "interview", "survey", "focus group", "consent", "ethic",
           "kaitiaki", "wahi tapu", "rohe", "runanga", "ngati", "ngai", "tainui"]
ALWAYS_KEEP = ("method", "participant", "data", "research design", "result", "finding",
               "ethic", "acknowledg", "funding", "availability")

def keyword_hits(c: Chunk) -> dict:
    f = fold(c.text)
    return {k: f.count(k) for k in LEXICON if k in f}

def select_chunks(chunks, budget=CONTEXT_BUDGET_CHARS):
    scored = []
    for c in chunks:
        hits = keyword_hits(c)
        must = any(k in c.section for k in ALWAYS_KEEP) or c.page <= 6
        if hits or must:
            scored.append((sum(hits.values()) + (100 if must else 0), c))
    scored.sort(key=lambda x: -x[0])
    picked, used = [], 0
    for _, c in scored:
        if used + len(c.text) <= budget:
            picked.append(c); used += len(c.text)
    return sorted(picked, key=lambda c: c.chunk_id)

for p in papers:
    p["selected"] = select_chunks(p["chunks"])
    (OUT / "chunks" / f"{p['paper_id']}_all.json").write_text(
        json.dumps([asdict(c) for c in p["chunks"]], ensure_ascii=False, indent=1))
    print(f"{p['paper_id']}: {len(p['selected'])}/{len(p['chunks'])} chunks selected, "
          f"{sum(len(c.text) for c in p['selected']):,} chars")

pd.DataFrame([{"paper": p["paper_id"], "chunk": c.chunk_id, "page": c.page, "section": c.section[:25],
               "top_hits": dict(sorted(keyword_hits(c).items(), key=lambda x: -x[1])[:5])}
              for p in papers for c in p["selected"]]).head(30)

p001: 11/11 chunks selected, 32,496 chars
p002: 17/58 chunks selected, 59,039 chars


,paper,chunk,page,section,top_hits
0,p001,c001,1,front,"{'maori': 12, 'wahine': 10, 'self-identif': 1,..."
1,p001,c002,3,front,"{'maori': 1, 'wahine': 1}"
2,p001,c003,3,literature review,"{'maori': 24, 'wahine': 8, 'self-identif': 4, ..."
3,p001,c004,4,literature review,"{'maori': 6, 'wahine': 2}"
4,p001,c005,6,literature review,"{'maori': 10, 'hapu': 1, 'whanau': 1, 'wahine'..."
5,p001,c006,7,literature review,{'maori': 3}
6,p001,c007,9,literature review,"{'maori': 5, 'whanau': 2, 'iwi': 1, 'hapu': 1,..."
7,p001,c008,10,literature review,{'maori': 6}
8,p001,c009,10,method,"{'maori': 13, 'wahine': 10, 'interview': 6, 'p..."
9,p001,c010,12,findings,"{'maori': 12, 'wahine': 7, 'participant': 4, '..."


## 6. Extraction schema and prompt

In [10]:
EvidenceType = Literal["participants", "ethnicity_variable", "maori_specific_results",
                       "iwi_hapu_data", "matauranga_or_cultural_material",
                       "governance_or_partnership", "context_mention"]

class Evidence(BaseModel):
    evidence_type: EvidenceType
    chunk_id: str
    section: str
    page: Optional[int] = None
    quote: str = Field(description="Verbatim text copied from the chunk, max 40 words")

class ExtractionResult(BaseModel):
    has_direct_participant_data: bool
    has_ethnicity_variable_only: bool
    has_maori_specific_analysis: bool
    has_iwi_hapu_data: bool
    has_matauranga_or_cultural_data: bool
    maori_mentioned_only_as_context: bool
    maori_sample_size: Optional[int] = None
    total_sample_size: Optional[int] = None
    data_sources: list[str]
    maori_led_or_partnered: Literal["documented", "not_stated", "unclear"]
    iwi_hapu_involvement: Literal["documented", "not_stated", "unclear"]
    ethics_statement_present: bool
    sensitivity_flags: list[str]
    evidence: list[Evidence]
    reason: str

PROMPT = """You are verifying whether a research output that was already classified as Māori-relevant
actually CONTAINS Māori data or Māori knowledge, and exactly what kind.

Definitions:
- participants: data were collected from Māori people (interviews, surveys, focus groups,
  observations, records). State sample sizes if given.
- ethnicity_variable: Māori appear only as an ethnicity category inside a wider dataset and
  are not analysed separately.
- maori_specific_results: results/findings are reported for Māori specifically.
- iwi_hapu_data: data about or supplied by specific iwi, hapū, marae or Māori organisations.
- matauranga_or_cultural_material: te reo, tikanga, mātauranga, whakapapa, kōrero tuku iho,
  Māori frameworks or cultural material that itself forms part of what is analysed.
- governance_or_partnership: Māori-led research, iwi partnership, kaupapa Māori methodology,
  Māori ethics/consultation or data-governance arrangements.
- context_mention: Māori appear only in background, literature review, policy context or
  recommendations.

Rules:
1. Do not infer Māori data merely because Māori are discussed. Set a flag to true only when
   the text shows data were collected from, about, or derived from Māori, or that Māori
   knowledge/cultural material itself is part of the research data.
2. Every true flag must be supported by at least one evidence item of the matching type.
3. Quotes must be copied VERBATIM from the chunk named in chunk_id, max 40 words.
   Never paraphrase inside "quote".
4. Do not reproduce participant-level, identifying, whakapapa, wāhi tapu location or
   restricted mātauranga content. Describe such material at study level and add a
   sensitivity_flags entry instead.
5. If ethics approval is not mentioned in the provided text, set ethics_statement_present=false.
6. If unsure, set the flag to false and explain in "reason".

PAPER TITLE: {title}

TEXT CHUNKS (each labelled with chunk_id, section and page):
{chunks}
"""

def build_prompt(p) -> str:
    body = "\n\n".join(f"[{c.chunk_id} | section: {c.section} | page {c.page}]\n{c.text}"
                        for c in p["selected"])
    return PROMPT.format(title=p["title"], chunks=body)

print(build_prompt(papers[0])[:1500], "...")

You are verifying whether a research output that was already classified as Māori-relevant
actually CONTAINS Māori data or Māori knowledge, and exactly what kind.

Definitions:
- participants: data were collected from Māori people (interviews, surveys, focus groups,
  observations, records). State sample sizes if given.
- ethnicity_variable: Māori appear only as an ethnicity category inside a wider dataset and
  are not analysed separately.
- maori_specific_results: results/findings are reported for Māori specifically.
- iwi_hapu_data: data about or supplied by specific iwi, hapū, marae or Māori organisations.
- matauranga_or_cultural_material: te reo, tikanga, mātauranga, whakapapa, kōrero tuku iho,
  Māori frameworks or cultural material that itself forms part of what is analysed.
- governance_or_partnership: Māori-led research, iwi partnership, kaupapa Māori methodology,
  Māori ethics/consultation or data-governance arrangements.
- context_mention: Māori appear only in background, l

## 7. Gemini structured extraction

Uses `response_schema` so output is forced into the Pydantic schema; temperature 0 for repeatability. If no key is set, the pilot extractions for the two default papers are loaded instead (they were produced by following this same prompt and schema).

In [12]:
PILOT_EXTRACTIONS = {
 "Wāhine Māori entrepreneurs and business growth: The power of social and cultural capital": {
  "has_direct_participant_data": True, "has_ethnicity_variable_only": False,
  "has_maori_specific_analysis": True, "has_iwi_hapu_data": True,
  "has_matauranga_or_cultural_data": True, "maori_mentioned_only_as_context": False,
  "maori_sample_size": 10, "total_sample_size": 10,
  "data_sources": ["semi-structured interviews (round 1 of 3)"],
  "maori_led_or_partnered": "documented", "iwi_hapu_involvement": "not_stated",
  "ethics_statement_present": False,
  "sensitivity_flags": ["per-participant tribal (iwi) affiliation reported in Table 3",
                        "pseudonymous participant quotes with industry codes; small sample could allow re-identification"],
  "evidence": [
   {"evidence_type": "participants", "chunk_id": "c009", "section": "method", "page": 11,
    "quote": "The research incorporates initial findings from the first round of three interviews with 10 wāhine Māori entrepreneurs, who have experienced or are currently undergoing business growth."},
   {"evidence_type": "participants", "chunk_id": "c001", "section": "front", "page": 2,
    "quote": "Indigenous women who self-identify as Māori, are business owners and founders, and want to or have grown their businesses."},
   {"evidence_type": "maori_specific_results", "chunk_id": "c010", "section": "findings", "page": 12,
    "quote": "which show that social and cultural capital are important for personal and business growth among wāhine Māori entrepreneurs."},
   {"evidence_type": "iwi_hapu_data", "chunk_id": "c010", "section": "findings", "page": 12,
    "quote": "Table 3 provides a summary of all participants by industry code, tribal affiliation"},
   {"evidence_type": "matauranga_or_cultural_material", "chunk_id": "c010", "section": "findings", "page": 13,
    "quote": "All participants whakapapa (genealogy) Māori which is our unique form of cultural capital."},
   {"evidence_type": "governance_or_partnership", "chunk_id": "c009", "section": "method", "page": 10,
    "quote": "This study uses a kaupapa Māori theory, methods and research philosophy"}],
  "reason": "Primary qualitative data from 10 Māori women via interviews, with findings reported specifically for wāhine Māori entrepreneurs. Participant iwi affiliation is tabulated. Kaupapa Māori methodology; no ethics statement found."},
 "Māori provisions in plans: Mana whenua, mauri of water, and wahi tapu": {
  "has_direct_participant_data": False, "has_ethnicity_variable_only": False,
  "has_maori_specific_analysis": False, "has_iwi_hapu_data": False,
  "has_matauranga_or_cultural_data": True, "maori_mentioned_only_as_context": False,
  "maori_sample_size": None, "total_sample_size": None,
  "data_sources": ["extracts from council RMA plans and LTCCPs (documentary)"],
  "maori_led_or_partnered": "documented", "iwi_hapu_involvement": "unclear",
  "ethics_statement_present": False, "sensitivity_flags": [],
  "evidence": [
   {"evidence_type": "matauranga_or_cultural_material", "chunk_id": "c005", "section": "1. introduction", "page": 6,
    "quote": "The extracts in this Report 3 on Māori Provisions in Plans relate to the three tikanga in the Ngā Mahi report: Mana Whenua, Mauri (relating to water), and Wāhi Tapu."},
   {"evidence_type": "matauranga_or_cultural_material", "chunk_id": "c007", "section": "1. introduction", "page": 4,
    "quote": "Customary rights, responsibilities and intimate relationships of Tangata Whenua with the natural and physical resources of the Auckland Region have been developed over several centuries."},
   {"evidence_type": "context_mention", "chunk_id": "c005", "section": "1. introduction", "page": 6,
    "quote": "Extracts in this current report are included from various council plans that we reviewed when developing our kauapa Māori outcomes and indicators."},
   {"evidence_type": "governance_or_partnership", "chunk_id": "c003", "section": "front", "page": 3,
    "quote": "Richard Jefferies of Ngāti Tukorehe and his firm, KCSM Consultancy Solutions Ltd were brought onto the PUCM research programme in 2002 to lead the Māori component of the research."},
   {"evidence_type": "governance_or_partnership", "chunk_id": "c003", "section": "front", "page": 3,
    "quote": "Nathan Kennedy, an environmental officer for Ngāti Whanaunga iwi"}],
  "reason": "No data collected from Māori people and no Māori-specific analysis. A compilation of council plan provisions organised by three tikanga, so the content is Māori cultural and governance material drawn from documents. Research was Māori-led within a kaupapa Māori framework."}
}

def gemini_extract(p) -> dict:
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=GEMINI_API_KEY)
    for attempt in range(5):
        try:
            resp = client.models.generate_content(
                model=GEMINI_MODEL, contents=build_prompt(p),
                config=types.GenerateContentConfig(response_mime_type="application/json",
                                                   response_schema=ExtractionResult,
                                                   temperature=0))
            return ExtractionResult.model_validate_json(resp.text).model_dump()
        except Exception as e:
            wait = 2 ** attempt * 5
            print(f"  retry {attempt+1} in {wait}s: {e}"); time.sleep(wait)
    raise RuntimeError("Gemini failed after retries")

for p in papers:
    if USE_GEMINI:
        print("Gemini →", p["paper_id"], p["title"][:60])
        p["extraction"], p["extractor"] = gemini_extract(p), GEMINI_MODEL
        if p["title"] in PILOT_EXTRACTIONS:
            p["pilot_extraction"] = PILOT_EXTRACTIONS[p["title"]]
    elif p["title"] in PILOT_EXTRACTIONS:
        p["extraction"], p["extractor"] = PILOT_EXTRACTIONS[p["title"]], "pilot (manual)"
        print("pilot extraction loaded →", p["paper_id"])
    else:
        p["extraction"], p["extractor"] = None, None
        print("⚠ no extraction for", p["paper_id"], "— set GEMINI_API_KEY")

Gemini → p001 Wāhine Māori entrepreneurs and business growth: The power of
  retry 1 in 5s: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/get-started).', 'status': 'NOT_FOUND'}}
  retry 2 in 10s: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/get-started).', 'status': 'NOT_FOUND'}}


KeyboardInterrupt: 

## 8. Evidence validation

Every quote is fuzzy-matched against the chunk it claims to come from. Quotes below the threshold are dropped, and any positive flag left without supporting evidence is downgraded and sent to review.

In [ ]:
FLAG_SUPPORT = {
    "has_direct_participant_data":     {"participants"},
    "has_maori_specific_analysis":     {"maori_specific_results"},
    "has_iwi_hapu_data":               {"iwi_hapu_data"},
    "has_matauranga_or_cultural_data": {"matauranga_or_cultural_material"},
    "has_ethnicity_variable_only":     {"ethnicity_variable"},
}

def validate(result: dict, chunks: list[Chunk], threshold=QUOTE_MATCH_THRESHOLD) -> dict:
    r = copy.deepcopy(result)
    by_id = {c.chunk_id: fold(c.text) for c in chunks}
    kept, dropped = [], []
    for ev in r.get("evidence", []):
        src = by_id.get(ev.get("chunk_id"), "")
        score = fuzz.partial_ratio(fold(ev["quote"]), src) if src else 0
        (kept if score >= threshold else dropped).append({**ev, "match_score": round(score)})
    r["evidence"], r["dropped_evidence"], r["unsupported_flags"] = kept, dropped, []
    for flag, types in FLAG_SUPPORT.items():
        if r.get(flag) and not any(e["evidence_type"] in types for e in kept):
            r[flag] = False; r["unsupported_flags"].append(flag)
    return r

for p in papers:
    if p["extraction"]:
        p["validated"] = validate(p["extraction"], p["selected"])
        v = p["validated"]
        print(f"{p['paper_id']}: kept {len(v['evidence'])}, dropped {len(v['dropped_evidence'])}, "
              f"unsupported {v['unsupported_flags']}")

## 9. Decision rules

| Category | Meaning |
|---|---|
| `DIRECT_MAORI_DATA` | Data collected from Māori, Māori-specific results, or iwi/hapū data |
| `MAORI_KNOWLEDGE_OR_CULTURAL_DATA` | Mātauranga, tikanga, te reo or cultural material is itself the research material |
| `ETHNICITY_VARIABLE_ONLY` | Māori are in the dataset but not analysed separately |
| `MAORI_CONTEXT_ONLY` | Māori appear only in background / policy / recommendations |
| `NO_MAORI_EVIDENCE` | Stage 2.1 likely a false positive |

Route: **CONFIRM** needs ≥2 validated non-context evidence items and no dropped quotes; everything else goes to **REVIEW**. Downgrades from Stage 2.1 always go to review. Any sensitivity flag adds **governance review**.

In [ ]:
def decide(r: dict) -> dict:
    core = [e for e in r["evidence"] if e["evidence_type"] != "context_mention"]
    if r["has_direct_participant_data"] or r["has_iwi_hapu_data"] or r["has_maori_specific_analysis"]:
        cat = "DIRECT_MAORI_DATA"
    elif r["has_matauranga_or_cultural_data"]:
        cat = "MAORI_KNOWLEDGE_OR_CULTURAL_DATA"
    elif r["has_ethnicity_variable_only"]:
        cat = "ETHNICITY_VARIABLE_ONLY"
    elif r["maori_mentioned_only_as_context"]:
        cat = "MAORI_CONTEXT_ONLY"
    else:
        cat = "NO_MAORI_EVIDENCE"

    if r["unsupported_flags"] or r["dropped_evidence"]:
        route, why = "REVIEW", "evidence failed validation"
    elif cat in ("DIRECT_MAORI_DATA", "MAORI_KNOWLEDGE_OR_CULTURAL_DATA") and len(core) >= 2:
        route, why = "CONFIRM", "≥2 validated evidence items"
    elif cat in ("MAORI_CONTEXT_ONLY", "NO_MAORI_EVIDENCE"):
        route, why = "REVIEW", "downgrade from Stage 2.1 needs human check"
    else:
        route, why = "REVIEW", "insufficient evidence"
    return {"category": cat, "route": route, "route_reason": why,
            "governance_review": bool(r.get("sensitivity_flags"))}

for p in papers:
    if p.get("validated"):
        p["decision"] = decide(p["validated"])
        d = p["decision"]
        print(f"{p['paper_id']}: {d['category']:<34} {d['route']:<8} "
              f"{'⚑ governance review' if d['governance_review'] else ''}  | {p['title'][:55]}")

### Tamper test — does the validator catch a fabricated quote?

In [ ]:
target = next((p for p in papers if p.get("validated") and p["validated"]["has_direct_participant_data"]), None)
if target:
    fake = copy.deepcopy(target["extraction"])
    fake["evidence"] = [e for e in fake["evidence"] if e["evidence_type"] != "maori_specific_results"]
    fake["evidence"].append({"evidence_type": "maori_specific_results",
                             "chunk_id": target["selected"][-1].chunk_id, "section": "findings", "page": 1,
                             "quote": "Māori participants increased their revenue by 40% through cultural networks."})
    fake["has_maori_specific_analysis"] = True
    fv = validate(fake, target["selected"])
    print("dropped:", [(d["evidence_type"], d["match_score"]) for d in fv["dropped_evidence"]])
    print("unsupported flags:", fv["unsupported_flags"])
    print("route:", decide(fv)["route"])

## 10. Evidence view

In [ ]:
for p in papers:
    if not p.get("validated"): continue
    d, v = p["decision"], p["validated"]
    print("=" * 100)
    print(f"{p['paper_id']} | {p['title']}\nhandle {p['handle']} | {p['rc_type']} | extractor: {p['extractor']}")
    print(f"→ {d['category']} | {d['route']} ({d['route_reason']})"
          + (" | ⚑ governance review" if d["governance_review"] else ""))
    print(f"sources: {v['data_sources']} | Māori n={v['maori_sample_size']} | "
          f"Māori-led: {v['maori_led_or_partnered']} | ethics stated: {v['ethics_statement_present']}")
    if v["sensitivity_flags"]: print("sensitivity:", v["sensitivity_flags"])
    print("reason:", v["reason"])
    for e in v["evidence"]:
        print(f"  [{e['match_score']:>3}] {e['evidence_type']:<32} p{e['page']:<3} “{e['quote'][:110]}”")
    for e in v["dropped_evidence"]:
        print(f"  ✗[{e['match_score']:>3}] {e['evidence_type']:<31} “{e['quote'][:110]}”")

## 11. Compare Gemini with the pilot extraction (only when Gemini ran on a pilot paper)

In [ ]:
FLAGS = ["has_direct_participant_data", "has_ethnicity_variable_only", "has_maori_specific_analysis",
         "has_iwi_hapu_data", "has_matauranga_or_cultural_data", "maori_mentioned_only_as_context"]
rows = []
for p in papers:
    if p.get("pilot_extraction") and p.get("validated"):
        pv = validate(p["pilot_extraction"], p["selected"])
        rows += [{"paper": p["paper_id"], "field": f, "gemini": p["validated"][f], "pilot": pv[f],
                  "agree": p["validated"][f] == pv[f]} for f in FLAGS]
        rows.append({"paper": p["paper_id"], "field": "CATEGORY", "gemini": p["decision"]["category"],
                     "pilot": decide(pv)["category"],
                     "agree": p["decision"]["category"] == decide(pv)["category"]})
pd.DataFrame(rows) if rows else print("Run with GEMINI_API_KEY to compare Gemini against the pilot.")

## 12. Save the Māori-data inventory

In [ ]:
records = []
for p in papers:
    if not p.get("validated"): continue
    v, d = p["validated"], p["decision"]
    records.append({
        "paper_id": p["paper_id"], "record_id_hash": p["record_id_hash"], "handle": p["handle"],
        "title": p["title"], "year": p["year"], "rc_type": p["rc_type"], "pdf_sha256": p["pdf_sha256"],
        "stage2_1_label": p["stage2_1_label"], "stage2_1_confidence": p["stage2_1_confidence"],
        "category": d["category"], "route": d["route"], "route_reason": d["route_reason"],
        "governance_review": d["governance_review"],
        **{f: v[f] for f in FLAGS},
        "maori_sample_size": v["maori_sample_size"], "total_sample_size": v["total_sample_size"],
        "data_sources": v["data_sources"], "maori_led_or_partnered": v["maori_led_or_partnered"],
        "iwi_hapu_involvement": v["iwi_hapu_involvement"],
        "ethics_statement_present": v["ethics_statement_present"],
        "sensitivity_flags": v["sensitivity_flags"], "n_evidence": len(v["evidence"]),
        "n_dropped": len(v["dropped_evidence"]), "evidence": v["evidence"],
        "reason": v["reason"], "extractor": p["extractor"],
        "prompt_sha": hashlib.sha256(PROMPT.encode()).hexdigest()[:12],
        "run_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
    })

(OUT / "inventory.json").write_text(json.dumps(records, ensure_ascii=False, indent=2))
inv = pd.DataFrame(records)
inv.drop(columns=["evidence"]).to_csv(OUT / "inventory.csv", index=False)
print("Saved:", OUT / "inventory.json", "and", OUT / "inventory.csv")
inv[["paper_id", "title", "category", "route", "governance_review", "n_evidence", "maori_sample_size"]]

In [ ]:
# Optional: download results from Colab
try:
    from google.colab import files
    import shutil
    shutil.make_archive("/content/stage2_2_results", "zip", OUT)
    files.download("/content/stage2_2_results.zip")
except Exception:
    pass

## Next steps

1. **Run with Gemini** on the two pilot papers and check Section 11 — if Gemini disagrees with the pilot, read the evidence to see who is right and tighten the prompt.
2. **Scale to 50**: set `PICK_TITLES = []`, `N_PAPERS = 50`. Mix in Stage 2.1 boundary cases (the GPT-OSS disagreements) and ~10 `non_maori_origin` records to estimate metadata misses.
3. **Review every output by hand** at 50, then start the human gold set (Māori and domain experts, two annotators per paper, Cohen's κ).
4. **Improve extraction** where the pilot showed weakness: GROBID for section headings in reports/theses; OCR (`ocrmypdf`) for scanned theses flagged `needs_ocr`.
5. **Store** in Postgres with runs versioned by `prompt_sha` and model, so prompt changes can be compared.